 # **1D Convolutional Neural Network**
A Convolutional Neural Network (CNN) is a technology used to detect and classify objects in images. It is modeled after human vision, just as regular neural networks mimic human neural circuits.

# CNN visual processing flow

CNN has a similar flow to human vision processing, where each step corresponds to the following elements:

- Simple cells : convolutional layers
- Complex Cells : Pooling Layers
CNN achieves image recognition by cooperating with convolutional layers and pooling layers to extract various features using filters with a multi-layered structure. The details of this are explained next.

# CNN processing flow
1. Input data: the input to CNN is two-dimensional image data. In reality, the input can be three-dimensional data, since it is processed in chunks called mini-batches.
2. Convolutional layer: The input image data is passed through the convolutional layer. In this layer, small filters called kernels are applied to the image to extract features. A kernel is a relatively small 2D matrix, and by using multiple kernels, various features are extracted. The output is called a "feature map."
3. Pooling layer: A pooling layer is applied to the feature map. Its role is to correct the misalignment and summarize the features. Pooling is typically done using the maximum or average value. This allows for stable feature extraction regardless of the size or position of the image.
4. Full connected layer: The features obtained from repeated convolution and pooling are fed into a fully connected layer, which converts the features into one-dimensional vectors that can be processed by traditional neural networks.
5. Output layer: Finally, the output according to the task is generated. For classification problems, there are nodes corresponding to each class, and they are labeled, for example, "dog" and "cat."

# Convolutional Layer
1. Kernels: Small matrices (weights) called "kernels" are applied to the input data. As the kernel slides over parts of the data, it performs a multiply-and-accumulate operation on each slide and passes the result to the next layer.
2. Bias: Bias also plays an important role in CNN. By adding bias to the convolution result, the model can express data more flexibly.
3. Padding: The convolution operation has the problem that the edges of the data are gradually lost. To address this issue, a technique called "padding" is used, which adds zeros around the data. Padding allows the size of the output data to be kept appropriate while still taking into account the edge information.
4. Stride: Stride is a value that indicates the interval at which the kernel is applied to the data. Changing the stride value affects the size of the output data.

# Pooling Layers
The pooling layer performs operations to obtain representative values ​​such as the maximum and average values. This has the following advantages:

- The model becomes more tolerant to variation and distortion because it extracts representative features and removes unnecessary noise.
- This reduces the amount of calculations and enables more efficient processing.
- The risk of overfitting is lowered because fewer parameters are used.
- 
Max pooling is common, but average pooling is also sometimes used. In practice, it is useful especially for highly skewed datasets. By increasing the stride, it is possible to further reduce the output size.

# Pooling Layers Applications
- Important element in practical model development to create more robust models e.g., it is effective in processing inputs with different deformations and scales in image recognition.
- Used in real-time applications that require improved computational efficiency and faster processing.

# Kernels
A kernel is a tool for detecting features in images and data. Kernels are used in calculations during convolution operations, which allows you to obtain specific patterns and features in an image. By providing many type. Kernels are frequently used in image processing, such as face recognition, object detection, etc. In practice, choosing the right kernel for a specific task is the key to improving accuracy.

# Channel
Image data is usually represented in three primary colors: red, green, and blue. A color image is formed by a set of three colors. This set of colors is called a "channel." Specifically, a color image has 3 channels. A grayscale image, or black and white image, has 1 channel. This is because black and white is not expressed using two colors, black and white, but using only different shades of black.

# 1D convolutional neural network
We will create a convolutional neural network (CNN) class from scratch. We will implement the algorithm using only a minimum number of libraries such as NumPy. In this unit, we will create a 1D convolutional layer and aim to understand the basics of convolution. In the next unit, we will create a 2D convolutional layer and a pooling layer to complete a CNN that is commonly used for images. The name of the class is Scratch1dCNNClassifier.
# Code Flow
- There are various types of activation functions, weight initialization methods, and gradient update methods. In order to flexibly use them according to the purpose, we define them as classes independently.
- Define the 1D convolutional layer class, which is the main topic of this textbook.
- Define a 1D convolutional neural network class using the classes defined in 1 and 2.
- Instantiate the class created in 3, fit関数pass data to it, and start learning.
- Once you have completed step 4, all that remains is to evaluate and visualize the data.

In [1]:
# Importing libraries
import numpy as np
import math
from keras.datasets import mnist
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

2025-04-22 19:10:19.308218: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745349019.603310      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745349019.688492      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Problem 1: Creating a one-dimensional convolutional layer class with the number of channels limited to 1

In [2]:
import numpy as np

class SimpleConv1d:
    def __init__(self, filter_size, input_size):
        # Initialize the layer with a given filter size and input size
        self.filter_size = filter_size
        self.input_size = input_size
        self.output_size = input_size - filter_size + 1  # No padding, stride 1
        
        # Xavier initialization for weights
        self.weights = np.random.randn(filter_size) / np.sqrt(input_size)
        self.bias = np.zeros(1)
        
    def forward(self, x):
        """ Forward propagation """
        self.input = x  # Save input for backpropagation
        self.output = np.zeros(self.output_size)
        
        # Compute the output using the convolution operation
        for i in range(self.output_size):
            self.output[i] = np.sum(x[i:i + self.filter_size] * self.weights) + self.bias
            
        return self.output
    
    def backward(self, da):
        """ Backpropagation """
        # Initialize gradients
        dL_dw = np.zeros(self.filter_size)
        dL_db = np.zeros(1)
        dL_dx = np.zeros(self.input_size)
        
        # Compute gradients with respect to weights and bias
        for i in range(self.output_size):
            dL_db += da[i]
            for s in range(self.filter_size):
                dL_dw[s] += da[i] * self.input[i + s]
        
        # Compute gradients with respect to the input
        for j in range(self.input_size):
            for s in range(self.filter_size):
                if 0 <= (j - s) < self.output_size:
                    dL_dx[j] += da[j - s] * self.weights[s]
        
        return dL_dw, dL_db, dL_dx
    
    def update_params(self, dL_dw, dL_db, learning_rate):
        """ Update parameters using gradient descent """
        self.weights -= learning_rate * dL_dw
        self.bias -= learning_rate * dL_db


# Problem 2: Calculating the output size after 1-D convolution
We have seen that the size of the data after passing through a convolutional layer changes depending on the input and output values, but the formula for calculating the output size when taking padding and stride into account is as follows.

$$ N_{out} = \frac{N_{in} + 2P − F}{S} + 1 $$
$N_{out}$ : Output size (number of features)

$N_{in}$ : Input size (number of features)

$P$ : number of paddings in a direction

$F$ : filter size

$S$ : stride size

We create a function that performs this calculation.

In [3]:
def output_size_calculation(n_in, F, P=0, S=1):
    n_out = int((n_in + 2 * P - F) / S + 1)
    
    return n_out


# Problem 3: Experiment with 1D convolutional layers on small arrays
Check the forward and backpropagation for the following small array:

In [4]:
import numpy as np

def forward(x, w, b):
    F = len(w)  # Filter size
    N_in = len(x)  # Input size
    N_out = N_in - F + 1  # Output size
    
    # Use array indexing for efficient computation
    indexes = np.array([np.arange(i, i+F) for i in range(N_out)])
    a = np.sum(x[indexes] * w, axis=1) + b  # Convolution + bias
    
    return a

def backward(x, w, delta_a):
    F = len(w)
    N_out = len(delta_a)
    
    # Gradient of the bias
    delta_b = np.sum(delta_a)
    
    # Gradient of the weights
    delta_w = np.zeros_like(w)  # Correct the reference to the weights (use `w` instead of `self.weights`)
    for i in range(N_out):
        delta_w += delta_a[i] * x[i:i+F]  # Accumulate gradient properly
    
    # Gradient of the input
    delta_x = np.zeros(len(x))
    for i in range(N_out):
        delta_x[i:i+F] += delta_a[i] * w  # Accumulate gradients
    
    return delta_b, delta_w, delta_x

# Test the implementation
x = np.array([1, 2, 3, 4])
w = np.array([3, 5, 7])
b = np.array([1])

# Forward pass
a = forward(x, w, b)
print("Output (Forward pass):", a)

# Assume delta_a (gradient of the loss w.r.t output) from the next layer
delta_a = np.array([10, 20])

# Backward pass
delta_b, delta_w, delta_x = backward(x, w, delta_a)
print("Gradient of bias:", delta_b)
print("Gradient of weights:", delta_w)
print("Gradient of input:", delta_x)


Output (Forward pass): [35 50]
Gradient of bias: 30
Gradient of weights: [ 50  80 110]
Gradient of input: [ 30. 110. 170. 140.]


# Problem 4: Creating a 1D convolutional layer class with no limit on the number of channels
We create a class Conv1d for a one-dimensional convolutional layer that does not limit the number of channels to one.

In [5]:
import numpy as np

class Conv1d:
    def __init__(self, C_out, C_in, F, P=0):

        self.C_out = C_out  # Number of output channels
        self.C_in = C_in    # Number of input channels
        self.F = F          # Filter size
        self.P = P          # Padding size
        
        # Initialize weights with Xavier initialization
        self.weights = np.random.randn(C_out, C_in, F) / np.sqrt(C_in * F)
        self.bias = np.zeros(C_out)
    
    def forward(self, x):

        N_in = x.shape[1]  # Number of input features
        N_padded = N_in + 2 * self.P  # Input size after padding
        N_out = N_padded - self.F + 1  # Number of output features after convolution
        
        # Apply padding (zero-padding by default)
        x_padded = np.pad(x, ((0, 0), (self.P, self.P)), mode='constant', constant_values=0)
        
        a = np.zeros((self.C_out, N_out))  # Initialize output
        
        for c_out in range(self.C_out):  # Loop over output channels
            for c_in in range(self.C_in):  # Loop over input channels
                for i in range(N_out):  # Loop over output positions
                    a[c_out, i] += np.sum(x_padded[c_in, i:i+self.F] * self.weights[c_out, c_in])  # Convolution step
            a[c_out] += self.bias[c_out]  # Add bias
        
        return a
    
    def backward(self, x, delta_a):

        N_in = x.shape[1]
        N_out = delta_a.shape[1]
        
        delta_b = np.sum(delta_a, axis=1)  # Gradient of the loss with respect to bias
        
        # Gradient of the loss with respect to the weights
        delta_w = np.zeros_like(self.weights)
        for c_out in range(self.C_out):
            for c_in in range(self.C_in):
                for i in range(N_out):
                    delta_w[c_out, c_in] += delta_a[c_out, i] * x[c_in, i:i+self.F].sum(axis=0)  # Gradient wrt to weight
        
        # Gradient of the loss with respect to the input
        delta_x = np.zeros_like(x, dtype=np.float64)  # Ensure delta_x is of type float64
        
        # Apply padding to x for backward pass as well
        x_padded = np.pad(x, ((0, 0), (self.P, self.P)), mode='constant', constant_values=0)
        
        for c_out in range(self.C_out):
            for i in range(N_out):
                delta_x[:, i:i+self.F] += delta_a[c_out, i] * self.weights[c_out]  # Gradient wrt input
        
        # Remove padding from delta_x (return only the relevant portion)
        delta_x = delta_x[:, self.P:-self.P] if self.P > 0 else delta_x
        
        return delta_b, delta_w, delta_x

# Test the implementation with padding
x = np.array([[1, 2, 3, 4], [2, 3, 4, 5]])  # Shape: (C_in=2, N_in=4)
w = np.ones((3, 2, 3))  # Shape: (C_out=3, C_in=2, F=3)
b = np.array([1, 2, 3])  # Shape: (C_out=3)

conv1d_layer = Conv1d(C_out=3, C_in=2, F=3, P=1)  # Use padding of 1

# Forward pass
a = conv1d_layer.forward(x)
print("Output (Forward pass):\n", a)

# Assume delta_a (gradient of the loss w.r.t output) from the next layer
delta_a = np.array([[10, 20], [10, 20], [10, 20]])  # Shape: (C_out=3, N_out=2)

# Backward pass
delta_b, delta_w, delta_x = conv1d_layer.backward(x, delta_a)
print("Gradient of bias (delta_b):", delta_b)
print("Gradient of weights (delta_w):\n", delta_w)
print("Gradient of input (delta_x):\n", delta_x)


Output (Forward pass):
 [[ 1.05903218  1.97235347  2.42255988  1.18229476]
 [ 0.27225079  0.52526259  0.87957241  0.53043031]
 [-1.35284146 -0.43241396 -0.06071949  3.65586818]]
Gradient of bias (delta_b): [30 30 30]
Gradient of weights (delta_w):
 [[[240. 240. 240.]
  [330. 330. 330.]]

 [[240. 240. 240.]
  [330. 330. 330.]]

 [[240. 240. 240.]
  [330. 330. 330.]]]
Gradient of input (delta_x):
 [[-2.8609919   8.90153416]
 [22.33815227 -0.13548421]]


# Problem 5: (Advanced) Implementing padding
We add padding to the convolutional layer. For one-dimensional arrays, we add n features before and after. The simplest padding is zero padding , which is common in CNNs, but other methods include repeating edge values.

Some frameworks allow us to specify that the size of the original input should be kept. This is useful. NumPy has a padding function.

In [6]:
import numpy as np

class Conv1dWithPadding:
    def __init__(self, C_out, C_in, F, P=0, padding_type='zero'):
  
        self.C_out = C_out  # Number of output channels
        self.C_in = C_in    # Number of input channels
        self.F = F          # Filter size
        self.P = P          # Padding size
        self.padding_type = padding_type  # Type of padding
        
        # Initialize weights with Xavier initialization
        self.weights = np.random.randn(C_out, C_in, F) / np.sqrt(C_in * F)
        self.bias = np.zeros(C_out)
    
    def forward(self, x):
        
        N_in = x.shape[1]  # Number of input features
        N_padded = N_in + 2 * self.P  # Input size after padding
        N_out = N_padded - self.F + 1  # Number of output features after convolution
        
        # Apply padding (either zero-padding or edge-padding)
        if self.padding_type == 'zero':
            x_padded = np.pad(x, ((0, 0), (self.P, self.P)), mode='constant', constant_values=0)
        elif self.padding_type == 'edge':
            x_padded = np.pad(x, ((0, 0), (self.P, self.P)), mode='edge')
        else:
            raise ValueError("Invalid padding type. Choose 'zero' or 'edge'.")
        
        a = np.zeros((self.C_out, N_out))  # Initialize output
        
        for c_out in range(self.C_out):  # Loop over output channels
            for c_in in range(self.C_in):  # Loop over input channels
                for i in range(N_out):  # Loop over output positions
                    a[c_out, i] += np.sum(x_padded[c_in, i:i+self.F] * self.weights[c_out, c_in])  # Convolution step
            a[c_out] += self.bias[c_out]  # Add bias
        
        return a
    
    def backward(self, x, delta_a):
 
        N_in = x.shape[1]
        N_out = delta_a.shape[1]
        
        delta_b = np.sum(delta_a, axis=1)  # Gradient of the loss with respect to bias
        
        # Gradient of the loss with respect to the weights
        delta_w = np.zeros_like(self.weights)
        for c_out in range(self.C_out):
            for c_in in range(self.C_in):
                for i in range(N_out):
                    delta_w[c_out, c_in] += delta_a[c_out, i] * x[c_in, i:i+self.F].sum(axis=0)  # Gradient wrt to weight
        
        # Gradient of the loss with respect to the input
        delta_x = np.zeros_like(x, dtype=np.float64)  # Ensure delta_x is of type float64
        
        # Apply padding to x for backward pass as well
        if self.padding_type == 'zero':
            x_padded = np.pad(x, ((0, 0), (self.P, self.P)), mode='constant', constant_values=0)
        elif self.padding_type == 'edge':
            x_padded = np.pad(x, ((0, 0), (self.P, self.P)), mode='edge')
        
        for c_out in range(self.C_out):
            for i in range(N_out):
                delta_x[:, i:i+self.F] += delta_a[c_out, i] * self.weights[c_out]  # Gradient wrt input
        
        # Remove padding from delta_x (return only the relevant portion)
        delta_x = delta_x[:, self.P:-self.P] if self.P > 0 else delta_x
        
        return delta_b, delta_w, delta_x

# Test the implementation with padding
x = np.array([[1, 2, 3, 4], [2, 3, 4, 5]])  # Shape: (C_in=2, N_in=4)
w = np.ones((3, 2, 3))  # Shape: (C_out=3, C_in=2, F=3)
b = np.array([1, 2, 3])  # Shape: (C_out=3)

conv1d_layer = Conv1dWithPadding(C_out=3, C_in=2, F=3, P=1, padding_type='edge')  # Use padding of 1 and edge padding

# Forward pass
a = conv1d_layer.forward(x)
print("Output (Forward pass):\n", a)

# Assume delta_a (gradient of the loss w.r.t output) from the next layer
delta_a = np.array([[10, 20], [10, 20], [10, 20]])  # Shape: (C_out=3, N_out=2)

# Backward pass
delta_b, delta_w, delta_x = conv1d_layer.backward(x, delta_a)
print("Gradient of bias (delta_b):", delta_b)
print("Gradient of weights (delta_w):\n", delta_w)
print("Gradient of input (delta_x):\n", delta_x)


Output (Forward pass):
 [[ 0.65291183  1.01141711  1.35035101  1.60041986]
 [-0.2188923  -0.2756567  -0.293122   -0.75243513]
 [-1.8243688  -3.23615053 -4.1445712  -3.96529848]]
Gradient of bias (delta_b): [30 30 30]
Gradient of weights (delta_w):
 [[[240. 240. 240.]
  [330. 330. 330.]]

 [[240. 240. 240.]
  [330. 330. 330.]]

 [[240. 240. 240.]
  [330. 330. 330.]]]
Gradient of input (delta_x):
 [[ -1.64485513  -5.12736885]
 [  6.57602721 -11.5036429 ]]


# Problem 6: (Advanced problem) Dealing with mini-batches
Up to this point, we have been using a batch size of 1. However, in reality, mini-batch learning is performed just like in the fully connected layer. Change the Conv1d class so that multiple data can be calculated simultaneously. 

In [7]:
import numpy as np

class Conv1dWithPaddingMiniBatch:
    def __init__(self, C_out, C_in, F, P=0):
        """
        Initialize the Conv1d layer with specified output channels, input channels, filter size, padding, and mini-batch support.
        
        Parameters:
        - C_out: Number of output channels (filters)
        - C_in: Number of input channels
        - F: Filter size
        - P: Padding size (default is 0 for no padding)
        """
        self.C_out = C_out  # Number of output channels
        self.C_in = C_in    # Number of input channels
        self.F = F          # Filter size
        self.P = P          # Padding size
        
        # Initialize weights with Xavier initialization
        self.weights = np.random.randn(C_out, C_in, F) / np.sqrt(C_in * F)
        self.bias = np.zeros(C_out)
    
    def forward(self, x):
        """
        Perform the forward pass of the 1D convolution with padding and mini-batch support.
        
        Parameters:
        - x: Input array of shape (batch_size, C_in, N_in)
        
        Returns:
        - a: Output array of shape (batch_size, C_out, N_out)
        """
        batch_size = x.shape[0]
        N_in = x.shape[2]  # Number of input features per sample
        N_padded = N_in + 2 * self.P  # Input size after padding
        N_out = N_padded - self.F + 1  # Number of output features after convolution
        
        # Apply padding (zero-padding by default)
        x_padded = np.pad(x, ((0, 0), (0, 0), (self.P, self.P)), mode='constant', constant_values=0)
        
        a = np.zeros((batch_size, self.C_out, N_out))  # Initialize output array for the batch
        
        for b in range(batch_size):  # Loop over each sample in the mini-batch
            for c_out in range(self.C_out):  # Loop over output channels
                for c_in in range(self.C_in):  # Loop over input channels
                    for i in range(N_out):  # Loop over output positions
                        a[b, c_out, i] += np.sum(x_padded[b, c_in, i:i+self.F] * self.weights[c_out, c_in])  # Convolution step
                a[b, c_out] += self.bias[c_out]  # Add bias to each output channel
        
        return a
    
    def backward(self, x, delta_a):
        """
        Perform the backward pass to compute gradients of the loss with respect to the weights, bias, and input,
        with mini-batch support.
        
        Parameters:
        - x: Input array of shape (batch_size, C_in, N_in)
        - delta_a: Gradient of the loss with respect to the output of shape (batch_size, C_out, N_out)
        
        Returns:
        - delta_b: Gradient of the loss with respect to the bias
        - delta_w: Gradient of the loss with respect to the weights
        - delta_x: Gradient of the loss with respect to the input
        """
        batch_size = x.shape[0]
        N_in = x.shape[2]
        N_out = delta_a.shape[2]
        
        delta_b = np.sum(delta_a, axis=(0, 2))  # Gradient of the loss with respect to bias (sum over batch and output length)
        
        # Gradient of the loss with respect to the weights
        delta_w = np.zeros_like(self.weights)
        for b in range(batch_size):  # Loop over each sample in the mini-batch
            for c_out in range(self.C_out):
                for c_in in range(self.C_in):
                    for i in range(N_out):
                        delta_w[c_out, c_in] += delta_a[b, c_out, i] * x[b, c_in, i:i+self.F]  # Accumulate gradients
        
        # Gradient of the loss with respect to the input
        delta_x = np.zeros_like(x, dtype=np.float64)  # Ensure delta_x is of type float64
        
        # Apply padding to x for backward pass as well
        x_padded = np.pad(x, ((0, 0), (0, 0), (self.P, self.P)), mode='constant', constant_values=0)
        
        for b in range(batch_size):  # Loop over each sample in the mini-batch
            for c_out in range(self.C_out):
                for i in range(N_out):
                    delta_x[b, :, i:i+self.F] += delta_a[b, c_out, i] * self.weights[c_out]  # Gradient wrt input
        
        # Remove padding from delta_x (return only the relevant portion)
        delta_x = delta_x[:, :, self.P:-self.P] if self.P > 0 else delta_x
        
        return delta_b, delta_w, delta_x

# Test the implementation with mini-batches
x = np.array([[[1, 2, 3, 4], [2, 3, 4, 5]],  # Batch size = 2, C_in = 2, N_in = 4
              [[2, 3, 4, 5], [3, 4, 5, 6]]])  # Batch size = 2, C_in = 2, N_in = 4
w = np.ones((3, 2, 3))  # C_out = 3, C_in = 2, F = 3
b = np.array([1, 2, 3])  # Biases for the 3 output channels

conv1d_layer = Conv1dWithPaddingMiniBatch(C_out=3, C_in=2, F=3, P=1)  # Use padding of 1

# Forward pass
a = conv1d_layer.forward(x)
print("Output (Forward pass):\n", a)

# Assume delta_a (gradient of the loss w.r.t output) from the next layer
delta_a = np.array([[[10, 20], [10, 20], [10, 20]],  # For the 1st sample
                    [[10, 20], [10, 20], [10, 20]]])  # For the 2nd sample

# Backward pass
delta_b, delta_w, delta_x = conv1d_layer.backward(x, delta_a)
print("Gradient of bias (delta_b):", delta_b)
print("Gradient of weights (delta_w):\n", delta_w)
print("Gradient of input (delta_x):\n", delta_x)

Output (Forward pass):
 [[[ 0.14971573  0.92215068  1.50273092  4.99328258]
  [ 0.09448478  0.60266737  0.90443057 -0.48276621]
  [ 0.38822305  1.46046371  2.31435557  1.4442539 ]]

 [[ 0.85746776  1.50273092  2.08331116  6.09316472]
  [-0.00795309  0.90443057  1.20619378 -0.51656448]
  [ 0.59286132  2.31435557  3.16824743  1.96588003]]]
Gradient of bias (delta_b): [60 60 60]
Gradient of weights (delta_w):
 [[[130. 190. 250.]
  [190. 250. 310.]]

 [[130. 190. 250.]
  [190. 250. 310.]]

 [[130. 190. 250.]
  [190. 250. 310.]]]
Gradient of input (delta_x):
 [[[13.67156827 18.83843506]
  [11.46836042 -4.12463943]]

 [[13.67156827 18.83843506]
  [11.46836042 -4.12463943]]]


# Problem 7: (Advanced Task) Any number of strides
So far we have implemented a stride that is limited to 1, but please make it possible to support any stride number. We modify the forward pass to account for different stride values.

In [8]:
import numpy as np

class SimpleConv1d:
    def __init__(self, filter_size, input_size, stride=1):
        self.filter_size = filter_size
        self.input_size = input_size
        self.stride = stride  # Add stride parameter
        self.output_size = (input_size - filter_size) // stride + 1  # Adjust output size for stride
        self.weights = np.random.randn(filter_size) / np.sqrt(input_size)  # Xavier initialization
        self.bias = np.zeros(1)
        
    def forward(self, x):
        """ Forward propagation """
        self.input = x  # Save input for backpropagation
        self.output = np.zeros(self.output_size)  # Initialize the output array
        
        # Compute the output using the convolution operation with strides
        for i in range(self.output_size):
            self.output[i] = np.sum(x[i * self.stride:i * self.stride + self.filter_size] * self.weights) + self.bias
            
        return self.output
    
    def backward(self, da):
        """ Backpropagation """
        dL_dw = np.zeros(self.filter_size)  # Gradient of weights
        dL_db = np.zeros(1)  # Gradient of bias
        dL_dx = np.zeros(self.input_size)  # Gradient of input
        
        # Compute gradients with respect to weights and bias
        for i in range(self.output_size):
            dL_db += da[i]
            for s in range(self.filter_size):
                dL_dw[s] += da[i] * self.input[i * self.stride + s]
        
        # Compute gradients with respect to the input
        for j in range(self.input_size):
            for s in range(self.filter_size):
                if 0 <= (j - s) < self.output_size * self.stride:
                    dL_dx[j] += da[(j - s) // self.stride] * self.weights[s]
        
        return dL_dw, dL_db, dL_dx
    
    def update_params(self, dL_dw, dL_db, learning_rate):
        """ Update parameters using gradient descent """
        self.weights -= learning_rate * dL_dw
        self.bias -= learning_rate * dL_db


# Problem 8: Learning and estimation
Replace some of the fully connected layers of the neural network we have used so far with Conv1d, train and estimate MNIST, and calculate the accuracy.

Please use the fully connected layer as is for the output layer. However, if there are multiple channels, you cannot input to the fully connected layer. Either make the channel at that stage 1, or smooth it.

Since one-dimensional convolution of images is not performed in practice, accuracy is not an issue.


In [9]:
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score

# Load and preprocess MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(X_train.shape[0], -1).astype(np.float32) / 255.0
X_test = X_test.reshape(X_test.shape[0], -1).astype(np.float32) / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Convolution layer (1D)
class SimpleConv1d:
    def __init__(self, filter_size, input_size, stride=1):
        self.filter_size = filter_size
        self.input_size = input_size
        self.stride = stride
        self.output_size = (input_size - filter_size) // stride + 1
        self.weights = np.random.randn(filter_size) / np.sqrt(input_size)
        self.bias = np.zeros(1)

    def forward(self, x):
        self.input = x
        batch_size = x.shape[0]
        output = np.zeros((batch_size, self.output_size))
        
        for b in range(batch_size):
            for i in range(self.output_size):
                output[b, i] = np.sum(
                    x[b, i * self.stride:i * self.stride + self.filter_size] * self.weights
                ) + self.bias.item()
        return output

    def backward(self, da):
        dL_dw = np.zeros(self.filter_size)
        dL_db = np.zeros(1)
        
        for b in range(da.shape[0]):
            for i in range(self.output_size):
                dL_db += da[b, i]
                for s in range(self.filter_size):
                    dL_dw[s] += da[b, i] * self.input[b, i * self.stride + s]
        dL_dx = None  # Not used in this implementation
        return dL_dw, dL_db, dL_dx

    def update_params(self, dL_dw, dL_db, learning_rate):
        self.weights -= learning_rate * dL_dw
        self.bias -= learning_rate * dL_db

# Custom CNN classifier
class ScratchCNNClassifier:
    def __init__(self, num_epoch=10, lr=0.01, batch_size=20, n_features=784, n_output=10, verbose=True):
        self.num_epoch = num_epoch
        self.lr = lr
        self.verbose = verbose
        self.batch_size = batch_size
        self.n_features = n_features
        self.n_output = n_output

        # Conv1D layer
        self.Conv1d = SimpleConv1d(filter_size=3, input_size=n_features, stride=1)

        # FC layer dimensions
        self.FC_weights = np.random.randn(self.Conv1d.output_size, n_output) * 0.01
        self.FC_bias = np.zeros(n_output)

    def forward_propagation(self, X):
        self.A1 = self.Conv1d.forward(X)  # shape: (batch_size, conv_output)
        self.A1_flattened = self.A1  # already flattened
        Z1 = np.dot(self.A1_flattened, self.FC_weights) + self.FC_bias  # shape: (batch_size, 10)
        return Z1

    def back_propagation(self, y_true, y_pred):
        batch_size = y_true.shape[0]

        # Mean Squared Error loss
        loss = np.sum((y_pred - y_true) ** 2) / batch_size
        dZ1 = 2 * (y_pred - y_true) / batch_size

        # Fully connected gradients
        dW_fc = np.dot(self.A1_flattened.T, dZ1)
        dB_fc = np.sum(dZ1, axis=0)

        # Backprop to Conv1d
        dA1_flattened = np.dot(dZ1, self.FC_weights.T)
        dA1 = dA1_flattened.reshape(self.A1.shape)
        dL_dw, dL_db, _ = self.Conv1d.backward(dA1)

        # Update weights
        self.FC_weights -= self.lr * dW_fc
        self.FC_bias -= self.lr * dB_fc
        self.Conv1d.update_params(dL_dw, dL_db, self.lr)

        return loss

    def fit(self, X_train, y_train):
        for epoch in range(self.num_epoch):
            total_loss = 0
            for i in range(0, len(X_train), self.batch_size):
                batch_X = X_train[i:i + self.batch_size]
                batch_y = y_train[i:i + self.batch_size]

                y_pred = self.forward_propagation(batch_X)
                loss = self.back_propagation(batch_y, y_pred)
                total_loss += loss

            if self.verbose:
                print(f"Epoch {epoch + 1}/{self.num_epoch}, Loss: {total_loss:.4f}")

    def predict(self, X):
        y_pred = self.forward_propagation(X)
        return np.argmax(y_pred, axis=1)

# Train and test
cnn = ScratchCNNClassifier(num_epoch=5, lr=0.01, batch_size=32)
cnn.fit(X_train, y_train)

y_pred = cnn.predict(X_test)
accuracy = accuracy_score(np.argmax(y_test, axis=1), y_pred)
print(f"Test Accuracy: {accuracy:.4f}")


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5, Loss: 871.2084
Epoch 2/5, Loss: 748.8528
Epoch 3/5, Loss: 740.5549
Epoch 4/5, Loss: 736.3522
Epoch 5/5, Loss: 733.6598
Test Accuracy: 0.8485


# Summary
To replace some of the fully connected layers with Conv1d layers in the neural network and train the model on the MNIST dataset, we made the following changes:

- Used Conv1d Layer: We  replaced some of the fully connected layers in the neural network with Conv1d layers
- The output of the Conv1d layer was flattened (reduced to a 1D vector) before being passed into the fully connected output layer.
- Flattening: Since Conv1d works with sequences, we flattenned the output from the convolutional layer to match the input shape required by the fully connected (FC) layer.
- Use Output Fully Connected Layer: The final output was produced by a fully connected layer, but the input came from the Conv1d layer.
- Handling Multiple Channels: number of channels were reduced to 1 before passing it to the fully connected layer, or use some form of pooling to smooth the channels and reduce them.
- Training on MNIST: used MNIST dataset, which consists of 28x28 pixelimages (flattened into 784 features)